# A2 — Pierce 1890 Knowledge-Base Demo
**Team G07 · doc-agent · Assignment 2**

This notebook provides graded A2 evidence:
1. **Part 1** — OCR quality metrics (CER, WER, Word-F1) on the 24 held-out pages vs. `grading_kit/labels.jsonl`
2. **Part 2** — Index overview (chunk count, dimension, pages indexed)
3. **Part 3** — Live vector retrieval demo with page citations and rendered figure images

> Run cells top to bottom after `bash scripts/build_index.sh` has completed.

In [ ]:
import sys, json, re, html
from collections import Counter
from pathlib import Path

import yaml
import numpy as np

REPO = Path('.').resolve()
sys.path.insert(0, str(REPO / 'src'))

cfg = yaml.safe_load((REPO / 'configs' / 'config.yaml').read_text())

LABELS_PATH   = REPO / 'grading_kit' / 'labels.jsonl'
INDEX_DIR     = REPO / Path(cfg['index']['path'])
CHANDRA_DIR   = REPO / 'chandra'
PAGES_MD      = REPO / 'chandra' / 'pages.md'
IMG_IDX_PATH  = INDEX_DIR / 'image_index.json'

print('REPO      :', REPO)
print('Labels    :', LABELS_PATH.exists())
print('Index dir :', INDEX_DIR.exists())


## Part 1 — OCR Quality on Held-Out Pages

In [ ]:
if not LABELS_PATH.is_file():
    raise FileNotFoundError(f'Missing {LABELS_PATH} — add hand-corrected labels.')

labels = {}
for line in LABELS_PATH.read_text('utf-8').splitlines():
    if not line.strip() or line.lstrip().startswith('#'):
        continue
    row = json.loads(line)
    labels[row['page_id']] = row['text']

print(f'Loaded {len(labels)} held-out labels: {sorted(labels.keys())[:5]} ...')


In [ ]:
# Parse chandra/pages.md and extract clean text per page
from doc_agent.index.chunk import load_from_pages_markdown

page_chunks, image_index = load_from_pages_markdown(PAGES_MD, cfg['ingest']['doc_id'])
chandra_text = {c.page_ids[0]: c.text for c in page_chunks}
print(f'Total pages indexed: {len(chandra_text)}')

# ─── Metric helpers ───────────────────────────────────────────────────────────
def normalize(t): return re.sub(r'\s+', ' ', t).strip().lower()

def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b)+1))
    for i, av in enumerate(a, 1):
        cur = [i]
        for j, bv in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[-1]+1, prev[j-1]+(av!=bv)))
        prev = cur
    return prev[-1]

def word_f1(hyp, ref):
    h, r = Counter(normalize(hyp).split()), Counter(normalize(ref).split())
    tp = sum((h & r).values())
    if not h and not r: return 1.0
    if not tp: return 0.0
    p, rc = tp/sum(h.values()), tp/sum(r.values())
    return 2*p*rc/(p+rc)

# ─── Score each held-out page ────────────────────────────────────────────────
results = []
for pid, ref in sorted(labels.items()):
    hyp  = chandra_text.get(pid, '')
    ref_n, hyp_n = normalize(ref), normalize(hyp)
    cer = levenshtein(list(hyp_n), list(ref_n)) / max(len(ref_n), 1)
    wer = levenshtein(hyp_n.split(), ref_n.split()) / max(len(ref_n.split()), 1)
    f1  = word_f1(hyp, ref)
    results.append({'page_id': pid, 'cer': cer, 'wer': wer, 'word_f1': f1})

macro_f1 = sum(r['word_f1'] for r in results) / max(len(results), 1)
micro_cer = sum(r['cer'] for r in results) / max(len(results), 1)
micro_wer = sum(r['wer'] for r in results) / max(len(results), 1)

print(f'\n{"Page":<10} {"CER":>7} {"WER":>7} {"Word F1":>9}')
print('-' * 38)
for r in results:
    print(f"{r['page_id']:<10} {r['cer']:>7.4f} {r['wer']:>7.4f} {r['word_f1']:>9.4f}")
print('-' * 38)
print(f'  MACRO AVG  {micro_cer:>7.4f} {micro_wer:>7.4f} {macro_f1:>9.4f}')
print(f'\nSample size: {len(results)} held-out pages')


## Part 2 — Index Overview

In [ ]:
from doc_agent.index.store import load

faiss_index, indexed_chunks, metadata = load(cfg)

# Load image index
img_idx = json.loads(IMG_IDX_PATH.read_text()) if IMG_IDX_PATH.exists() else {}
pages_with_figs = len(img_idx)
total_figs      = sum(len(v) for v in img_idx.values())

print('=' * 55)
print('  STAGE 4 — FAISS KNOWLEDGE BASE STATISTICS')
print('=' * 55)
print(f'  Index type         : {metadata["index_type"]}')
print(f'  Embedding model    : {cfg["embed"]["model"]}')
print(f'  Embedding dim      : {metadata["dimension"]}')
print(f'  Total chunks       : {metadata["count"]}')
print(f'  Chunk size (tokens): {cfg["index"]["chunk_tokens"]} (overlap {cfg["index"]["overlap"]})')
print(f'  Pages indexed      : {len(set(pid for c in indexed_chunks for pid in c.page_ids))}')
print(f'  Pages with figures : {pages_with_figs}  ({total_figs} total figure refs)')
print(f'  Index vectors      : {faiss_index.ntotal}')
print('=' * 55)


## Part 3 — Live Vector Retrieval Demo

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from IPython.display import display, Image as IPImage, Markdown

embedder = SentenceTransformer(cfg['embed']['model'])

def retrieve(query: str, k: int = 5):
    """Embed query, search index, return top-k (chunk, score, images) tuples."""
    qvec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, positions = faiss_index.search(qvec, k)
    hits = []
    for score, pos in zip(scores[0], positions[0]):
        if pos < 0: continue
        chunk = indexed_chunks[pos]
        page_id = chunk.page_ids[0]
        page_imgs = img_idx.get(page_id, [])
        hits.append({'chunk': chunk, 'score': float(score), 'images': page_imgs})
    return hits

def show_results(query: str, k: int = 3):
    display(Markdown(f'### Query: *{query}*'))
    hits = retrieve(query, k)
    for i, h in enumerate(hits, 1):
        c = h['chunk']
        display(Markdown(
            f'**Result {i}** | Page: `{c.page_ids[0]}` | Cosine score: `{h["score"]:.4f}`\n\n'
            f'> {c.text[:400]}...'
        ))
        for img_meta in h['images'][:2]:  # show up to 2 figures per page
            img_path = CHANDRA_DIR / img_meta['webp']
            if img_path.exists():
                print(f'  📷 {img_meta["caption"][:100]}')
                display(IPImage(str(img_path), width=450))
        print()

# ─── Demo queries ──────────────────────────────────────────────────────────
show_results('What are the superficial muscles of the chest and abdomen?')


In [ ]:
show_results('What remedies are prescribed for inflammation and fever?')


In [ ]:
show_results('Describe the structure and function of the human heart.')


## A2 Form — Numbers to Copy
Copy the values from the cells above directly into **Section 5** of `forms/A2_form.docx`.

| Metric | Value |
|---|---|
| OCR sample size | *from Part 1 output* |
| Macro Word-F1 | *from Part 1 output* |
| Micro CER | *from Part 1 output* |
| Chunks indexed | *from Part 2 output* |
| Embedding dim | 384 (all-MiniLM-L6-v2) |
| Index type | faiss:flat_ip |
| Pages with figures | *from Part 2 output* |
| Query shown | *from Part 3 output* |
| Top-1 page citation | *from Part 3 output* |